In [ ]:
-- Доля клиентов, у которых в текущем месяце повторился хотя бы один номинал
-- из предыдущего месяца

with prepared as (
    select
        client_id,
        date_trunc('month', mrc_start_date)::date as month_dt,

        case
            when lower(com_cus_sgr_desc) like '%200%' then 200
            when lower(com_cus_sgr_desc) like '%300%' then 300
            when lower(com_cus_sgr_desc) like '%400%' then 400
            when lower(com_cus_sgr_desc) like '%500%' then 500
            else null
        end as nominal_amount
    from cvm_sbx.YOUR_TABLE
    where client_id is not null
),

client_month_nominals as (
    select distinct
        client_id,
        month_dt,
        nominal_amount
    from prepared
    where nominal_amount is not null
),

client_month_compare as (
    select
        cur.client_id,
        cur.month_dt,
        max(
            case
                when prev.nominal_amount is not null then 1
                else 0
            end
        ) as has_same_nominal
    from client_month_nominals cur
    left join client_month_nominals prev
        on prev.client_id = cur.client_id
       and prev.month_dt = cur.month_dt - interval '1 month'
       and prev.nominal_amount = cur.nominal_amount
    where cur.month_dt > date '2025-12-01'
    group by
        cur.client_id,
        cur.month_dt
)

select
    month_dt,
    count(distinct client_id) as clients_cnt,
    sum(has_same_nominal) as clients_same_nominal_cnt,
    round(
        sum(has_same_nominal) * 100.0 / count(distinct client_id),
        2
    ) as same_nominal_pct
from client_month_compare
group by month_dt
order by month_dt;